# Análisis Detallado de Resultados MMLU (Falcon Mamba)
Este notebook analiza el rendimiento del modelo bajo distintos *prompts* emocionales.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from IPython.display import display

plt.style.use('ggplot')
sns.set_theme(style="whitegrid")

## 1. Carga de Datos
Cargamos los resultados de los archivos CSV correspondientes a cada emoción.

In [ ]:
data_dir = os.path.join('data_falcon_mamba', 'results_falcon_mamba')
emotions = ['original', 'anger', 'anxiety', 'courtesy', 'optimism']
dfs = []

for emotion in emotions:
    file_path = os.path.join(data_dir, f'results_{emotion}.csv')
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        df['prompt_type'] = emotion
        dfs.append(df)
    else:
        print(f'Archivo no encontrado: {file_path}')

if dfs:
    df_all = pd.concat(dfs, ignore_index=True)
    # Limpiar y extraer la letra predicha (predicted_letter)
    def clean_letter(val):
        if not isinstance(val, str):
            return None
        val = val.strip()
        if not val:
            return None
        first = val[0].upper()
        return first if first in ['A', 'B', 'C', 'D'] else None
    df_all['predicted_letter'] = df_all['predicted_answer'].apply(clean_letter)
    print(f'Total de registros cargados: {len(df_all)}')
else:
    print('No se pudieron cargar los datos. Verifica la ruta de data_dir.')

## 2. Precisión General por Emoción
Comparamos la precisión total (accuracy) de cada tipo de prompt.

In [ ]:
if 'df_all' in locals() and not df_all.empty:
    overall_acc = df_all.groupby('prompt_type')['is_correct'].mean() * 100
    overall_acc = overall_acc.sort_values(ascending=False)

    plt.figure(figsize=(10, 6))
    ax = sns.barplot(x=overall_acc.index, y=overall_acc.values, palette='viridis')
    plt.title('Precisión General por Tipo de Prompt (MMLU - Falcon Mamba)', fontsize=16)
    plt.ylabel('Precisión (%)', fontsize=12)
    plt.xlabel('Tipo de Prompt (Emoción)', fontsize=12)
    plt.ylim(0, max(overall_acc) + 10 if not overall_acc.empty else 100)

    for p in ax.patches:
        ax.annotate(f'{p.get_height():.2f}%', (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='baseline', fontsize=12, color='black', xytext=(0, 5),
                    textcoords='offset points')
    plt.show()

## 3. Precisión por Materia (Top 15 Variabilidad)
Analizamos cómo afecta cada prompt a las diferentes materias del MMLU. Mostramos las materias donde hay mayor variación de rendimiento.

In [ ]:
if 'df_all' in locals() and not df_all.empty:
    subject_acc = df_all.groupby(['subject', 'prompt_type'])['is_correct'].mean().unstack() * 100
    subject_acc['std_dev'] = subject_acc.std(axis=1)

    # Seleccionar las 15 materias con mayor variabilidad entre prompts
    top_var_subjects = subject_acc.sort_values('std_dev', ascending=False).head(15)
    top_var_subjects = top_var_subjects.drop(columns=['std_dev'])

    plt.figure(figsize=(12, 8))
    sns.heatmap(top_var_subjects, annot=True, fmt=".1f", cmap='YlGnBu', linewidths=.5)
    plt.title('Top 15 Materias con Mayor Variabilidad según el Prompt (%) (Falcon Mamba)', fontsize=16)
    plt.ylabel('Materia', fontsize=12)
    plt.xlabel('Tipo de Prompt', fontsize=12)
    plt.show()

## 4. Distribución de Respuestas por Letra
Observamos si algún prompt induce sesgos hacia responder más a una letra en particular.

In [ ]:
if 'df_all' in locals() and not df_all.empty:
    plt.figure(figsize=(10, 6))
    sns.countplot(data=df_all, x='predicted_letter', hue='prompt_type', palette='Set2', order=['A', 'B', 'C', 'D'])
    plt.title('Distribución de Letras Predichas por Tipo de Prompt (Falcon Mamba)', fontsize=16)
    plt.xlabel('Letra Predicha', fontsize=12)
    plt.ylabel('Cantidad de Respuestas', fontsize=12)
    plt.legend(title='Prompt', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

## 5. Análisis de Errores Inducidos por la Emoción
Identificamos preguntas que el modelo en estado original (neutral) pudo responder correctamente, pero bajo presión de ira falló.

In [ ]:
if 'df_all' in locals() and not df_all.empty:
    # Pivoteamos para ver el acierto por pregunta
    correct_pivot = df_all.pivot_table(index=['subject', 'question', 'target_answer'], 
                                       columns='prompt_type', values='is_correct', aggfunc='first')

    # Filtramos donde Original = True y Anger = False
    if 'original' in correct_pivot.columns and 'anger' in correct_pivot.columns:
        diff_cases = correct_pivot[(correct_pivot['original'] == True) & (correct_pivot['anger'] == False)]
        
        print(f"Total de preguntas donde Original acertó pero Anger falló: {len(diff_cases)}")
        display(diff_cases.head(10))
    else:
        print("No se encontraron ambas columnas ('original' y 'anger') para hacer la comparativa.")